## Q4 Writing Viterbi algorithm for the Nature Primer
Q- Write the Viterbi algorithm to implment Nature Primer.

Here are some suggestions:

a. You can begin by first defining all the parameters, such as states, transition matrix, and emmision matrix etc.

b. You can write a function to exactly calculate the values mentioned in the primer, for example, you can define a function get_log_prob_of_a_given_path ("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA"). This should output -41.22.

By doing the above two, you earn 1 mark.

Now, you have to implement this in real to get max likely path that would emmit the observed sequence. You MUST note that maximum likely path will just be Es, but that is okay. Implementation is the key.

### a) Defining paramenters- states, transition matrix, and emmision matrix as per Nature Primer

In [1]:
# Defining states and emitting states(does not include Start and End as they do not emit)
states = ['Start', 'E', '5', 'I', 'End']
emit_states = ['E', '5', 'I']

# Defining emissions as per Nature Primer
# For transitions and emissions with 0 probability, I have not defined them here as they will
# be taken 0 by defualt if I will try to access a key which is not in the dictionary
# as I will use get() method of dictionary in the code ahead to access the values
emissions_prob = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'G': 0.95,},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

# Defining transition probabilities as per Nature Primer
transitions_prob = {
    'Start': {'E': 1.0 },
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'End': 0.1},
    'End': {}
}

### b) Defining the function to get log probability of a given path

In [2]:
import math

def get_log_prob_of_a_given_path(state_path, obs_seq):
    total_log_prob = 0.0

    # Since we are calculating log probability so we can keep on adding each log value
    # (because log(p1*p1)=log(p1)+log(p2))

    # Transition from Start to first state
    total_log_prob += math.log(transitions_prob['Start'][state_path[0]])

    for i in range(len(state_path)):
        state = state_path[i] # this_stateent state
        obs_base = obs_seq[i] # this_stateent Observed base

        # Emission of the this_stateent observed base given this_stateent state
        emit_prob = emissions_prob[state].get(obs_base, 0)
        # If this probability is 0 then total probability would be 0 and log(0)=-infinity thus return -inf
        if emit_prob == 0:
            return float('-inf')
        total_log_prob += math.log(emit_prob)

        # Transition to next state if not at end of sequence
        if i < len(state_path) - 1:
            next_state = state_path[i + 1]
            trans_prob = transitions_prob[state].get(next_state, 0)
            # If this probability is 0 then total probability would be 0 and log(0)=-infinity thus return -inf
            if trans_prob == 0:
                return float('-inf')
            total_log_prob += math.log(trans_prob)
        else:
            # Transition from last state to End state
            trans_prob = transitions_prob[state].get('End', 0)
            if trans_prob == 0:
                return float('-inf')
            total_log_prob += math.log(trans_prob)
    # return the final log probability
    return round(total_log_prob, 2)

# Example run on the sequence in the Nature Primer
states="EEEEEEEEEEEEEEEEEE5IIIIIII"
sequence="CTTCATGTGAAAGCAGACGTAAGTCA"
print(f"States: {states}")
print(f"Sequence: {sequence}")
print(f"The log probability is {get_log_prob_of_a_given_path(states,sequence)}")

States: EEEEEEEEEEEEEEEEEE5IIIIIII
Sequence: CTTCATGTGAAAGCAGACGTAAGTCA
The log probability is -41.22


### c) Implementing Viterbi algorithm to get the max likely path for any given sequence

In [3]:
# Function implementing Viterbi algorithm to get the max likely path for any given sequence
def viterbi(obs_seq):
  # path is used to backtrack the max likely path
    backtracking_path = {}
  # dp is the dynamic programming table in form of array of dictionary
  # At the ith index of dp is a dictionary with states as keys with values as the probability of
  # reaching that state across max likely path
    dp = [{}]

    # Initialization from start to the first state
    for s in emit_states:
        if transitions_prob['Start'].get(s, 0) > 0 and emissions_prob[s].get(obs_seq[0], 0) > 0:
            dp[0][s] = math.log(transitions_prob['Start'][s]) + math.log(emissions_prob[s][obs_seq[0]])
            backtracking_path[s] = ['Start', s]
        else:
            dp[0][s] = float('-inf')
            backtracking_path[s] = ['Start']

    # Dynamic programming
    for t in range(1, len(obs_seq)):
        dp.append({})
        new_path = {}

        # Looping over states to fill dp[t]
        for this_state in emit_states:
            # Initialized to default values
            max_prob = float('-inf')
            best_prev = None

            # Looping over previous state to find maxm probability for current state
            for last_state in emit_states:
                if dp[t - 1][last_state] != float('-inf') and transitions_prob[last_state].get(this_state, 0) > 0:
                    prob = (dp[t - 1][last_state] +
                            math.log(transitions_prob[last_state][this_state]) +
                            math.log(emissions_prob[this_state].get(obs_seq[t], 1e-10)))
                    # Finding the max probability
                    if prob > max_prob:
                        max_prob = prob
                        best_prev = last_state

            # Storing
            dp[t][this_state] = max_prob
            new_path[this_state] = backtracking_path[best_prev] + [this_state] if best_prev else ['Start', this_state]

        backtracking_path = new_path

    # Transition to End state
    # Initializing
    max_final_prob = float('-inf')
    final_path = []
    for state in emit_states:
        if dp[-1][state] != float('-inf') and transitions_prob[state].get('End', 0) > 0:
            prob = dp[-1][state] + math.log(transitions_prob[state]['End'])
            if prob > max_final_prob:
                max_final_prob = prob
                final_path = backtracking_path[state] + ['End']

    # Returning the final path and the log probability
    return ''.join(final_path[1:-1]), round(max_final_prob, 2) # Sliced from 1:-1 to ignore Start and End states at the beginning and end respectively

observed_sequence=input("Enter the observed sequence ")
print(f"The entered sequence is {observed_sequence}")
path,log_prob=viterbi(observed_sequence)
print(f"The max likely path is {path[1:-1]}")
print(f"The corresponding max log probability is {log_prob}")

The entered sequence is CTTCATGTGAAAGCAGACGTAAGTCA
The max likely path is EEEEEEEEEEEEEEEEE5IIIIII
The corresponding max log probability is -41.22


The code in c part takes an input sequence fromt the user and calculates the max likely path and corresponding log probability and prints it.